In [1]:
import numpy as np
data = np.load('SOAP_data_with_H_ads_energy.npz')
descs = data['descs']
e_ads_per_h = data['e_ads_per_h']

In [2]:
print(descs.shape)
print(e_ads_per_h.shape)

(545, 780)
(545,)


In [3]:
X = descs
y = e_ads_per_h

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=22)

In [11]:
print(np.max(X), np.min(X))

1387.9950982850587 -292.1803865769172


In [6]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, BayesianRidge
# from sklearn.tree import DecisionTreeRegressor
# from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
# from sklearn.svm import SVR
# from sklearn.neighbors import KNeighborsRegressor
from sklearn.kernel_ridge import KernelRidge
# from xgboost import XGBRegressor

In [7]:
def screen_regression_models_on_train_set_using_cv(model_list:dict, X_train, y_train, compare_scaled=True):
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
    from sklearn.model_selection import KFold, cross_val_predict, cross_val_score, cross_validate
    model_list = list(model_dict.keys())
    mae_scores = []
    mse_scores = []
    rmse_scores =[]
    r2_scores_train = []
    r2_scores_test = []
    scorer = {'MAE': make_scorer(mean_absolute_error), 'MSE': make_scorer(mean_squared_error), 'RMSE': 'neg_root_mean_squared_error',
             'R2': make_scorer(r2_score)}
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    for key, model in model_dict.items():
        print(key)
        # Generate cross-validated predictions
        cv_scores = cross_validate(model, X_train, y_train, cv=kf, scoring=scorer, return_train_score=True)
        
        r2_scores_train.append(float(np.mean(cv_scores['train_R2'])))
        r2_scores_test.append(float(np.mean(cv_scores['test_R2'])))
        
    if compare_scaled:
        from sklearn.preprocessing import StandardScaler
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        r2_scores_scaled_train = []
        r2_scores_scaled_test = []
        for key, model in model_dict.items():
            print(key)
            
            # Generate cross-validated predictions
            cv_scores = cross_validate(model, X_train_scaled,
                                       y_train, cv=kf, scoring=scorer, return_train_score=True)

            r2_scores_scaled_train.append(float(np.mean(cv_scores['train_R2'])))
            r2_scores_scaled_test.append(float(np.mean(cv_scores['test_R2'])))

    model_performance = {'Model': model_list, 'R2 train': r2_scores_train, 
                         'R2 train scaled': r2_scores_scaled_train, 
                         'R2 test': r2_scores_test, 'R2 test scaled': r2_scores_scaled_test}
    return model_performance

In [8]:
lr = LinearRegression()
kernelreg = KernelRidge()
model_dict = {'Linear_Regression': lr, 'Kernel Ridge': kernelreg}

In [9]:
model_performance = screen_regression_models_on_train_set_using_cv(model_dict, X_train, y_train)

Linear_Regression
Kernel Ridge
Linear_Regression
Kernel Ridge


In [10]:
import pandas as pd
model_performance = pd.DataFrame(model_performance).sort_values(by='R2 train', ascending=False)
print(model_performance)

               Model      R2 train  R2 train scaled       R2 test  \
1       Kernel Ridge  1.934073e-10      -262.285191 -8.321393e-02   
0  Linear_Regression -4.539931e+22         0.000000 -4.367802e+22   

   R2 test scaled  
1     -318.407008  
0       -0.083214  


In [20]:
from sklearn.model_selection import GridSearchCV
best_parameters = {}
print(f'training started')
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)
# print(np.max(X_test_scaled), np.min(X_test_scaled))
param_grid = {'alpha': [0.5, 1, 2], 'degree': [1, 2, 3, 4]}
grid_search = GridSearchCV(KernelRidge(), param_grid, cv=5)
grid_search.fit(X_train, y_train)
# best_parameters = grid_search.best_params_
print(f"Best parameters: ", grid_search.best_params_)

training started
Best parameters:  {'alpha': 2, 'degree': 1}
